In [ ]:
EXPECTED_RUN = None  # Set only by ops.packaging after measured promotion gates.
EXPECTED_BUNDLE_SHA256 = None
EXPECTED_ARCHIVE_SHA256 = None


In [ ]:
import hashlib, json, subprocess, sys
from pathlib import Path
if not EXPECTED_RUN or not EXPECTED_BUNDLE_SHA256:
    raise RuntimeError("Build this notebook with the package stage; do not fill parameters manually")
import zipfile
def file_sha(path):
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1048576), b""):
            h.update(block)
    return h.hexdigest()
# Kaggle may expose uploaded ZIPs as archives or as expanded folders.
archives = [p for p in Path("/kaggle/input").rglob("biohub_payload.zip")
            if file_sha(p) == EXPECTED_ARCHIVE_SHA256]
if len(archives) > 1:
    raise RuntimeError("Ambiguous duplicate payload")
search_root = Path("/kaggle/input")
if archives:
    search_root = Path("/kaggle/working/biohub-package")
    search_root.mkdir(exist_ok=False)
    with zipfile.ZipFile(archives[0]) as archive:
        for info in archive.infolist():
            target = (search_root / info.filename).resolve()
            if search_root.resolve() not in target.parents or ((info.external_attr >> 16) & 0o170000) == 0o120000:
                raise RuntimeError("Unsafe archive entry")
        archive.extractall(search_root)
matches = []
for path in search_root.rglob("bundle.json"):
    if file_sha(path) == EXPECTED_BUNDLE_SHA256:
        matches.append(path)
if len(matches) != 1:
    raise RuntimeError("Expected exactly one matching immutable Biohub bundle")
bundle = matches[0].parent
spec = json.loads(matches[0].read_text())
if spec["run_id"] != EXPECTED_RUN:
    raise RuntimeError("Wrong experiment bundle")
if f"{sys.version_info.major}.{sys.version_info.minor}" != spec["python_minor"]:
    raise RuntimeError("Kaggle Python differs from wheelhouse target; rebuild compatible wheels before submission")
for name, expected in spec["file_sha256"].items():
    file = (bundle / name).resolve()
    if bundle.resolve() not in file.parents:
        raise RuntimeError("Invalid bundle relative path")
    if file_sha(file) != expected:
        raise RuntimeError("Bundle checksum mismatch: " + name)
wheels = [str(w) for w in sorted((bundle / "wheelhouse").glob("*.whl"))]
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", *wheels], check=True)
competition = "biohub-cell-tracking-during-development"
roots = [Path("/kaggle/input/competitions") / competition / "test",
         Path("/kaggle/input") / competition / "test"]
test = next((p for p in roots if p.is_dir()), None)
if test is None:
    raise RuntimeError("Missing competition test mount")
try:
    subprocess.run([sys.executable, str(bundle / "runtime/ops/kaggle_runtime.py"),
        "--bundle", str(bundle), "--test-root", str(test),
        "--output", "/kaggle/working/biohub-run"], check=True, timeout=40000)
except BaseException:
    # A CSV must never survive a failed/timed-out inference as a seemingly valid result.
    csv = Path("/kaggle/working/submission.csv")
    if csv.exists():
        csv.rename("/kaggle/working/failed-incomplete-submission.csv")
    raise
receipt = json.loads(Path("/kaggle/working/runtime_receipt.json").read_text())
assert receipt["status"] == "completed" and receipt["run_id"] == EXPECTED_RUN
print(json.dumps(receipt, indent=2))
